# DataLens model comparison

## tl;dr

This notebook is a thin experiment driver over the reusable `datalens.modeling` modules.
FY2024 is used for fitting and model selection.
FY2025 is scored only after the table-specific winners are locked.

## Context & Methods

Separate vendor and transaction anomaly models are compared against the deterministic baseline.
Isolation Forest is the first statistical model and Local Outlier Factor is the alternative.
Promotion requires non-inferiority to the deterministic baseline and preservation of deterministic critical findings.

### Key Assumptions

- Controlled defects are evaluation labels, not production fraud labels.
- Anomaly percentile is review evidence, not business severity or a calibrated defect probability.
- Statistical models may add review candidates but cannot remove deterministic critical findings.

## Data

Run the versioned project workflow and persist MLflow and comparison artifacts.

In [1]:
import pandas as pd

from datalens.modeling.workflow import run_modeling_experiment
from datalens.paths import ARTIFACTS_DIR

summary = run_modeling_experiment()
comparison_path = ARTIFACTS_DIR / "modeling" / "comparison-summary.json"
comparison_path

2026/06/22 13:19:00 INFO mlflow.store.db.utils: Creating initial MLflow database tables...


2026/06/22 13:19:00 INFO mlflow.store.db.utils: Updating database tables


2026/06/22 13:19:07 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


2026/06/22 13:19:13 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


2026/06/22 13:19:18 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


2026/06/22 13:19:22 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


WindowsPath('C:/Users/IBIKI/Desktop/Work/DataLens/artifacts/modeling/comparison-summary.json')

## Results

Summarize model selection, promotion, and the requested operational metrics.

In [2]:
summary["table_winners"], summary["promotion"]

({'transaction': 'local_outlier_factor', 'vendor': 'isolation_forest'},
 {'promoted': False,
  'gates': {'development_only_selection': True,
   'top_k_precision_non_inferior': False,
   'macro_f1_non_inferior': False,
   'false_alarm_rate_non_inferior': False,
   'deterministic_critical_findings_preserved': True},
  'criteria': {'minimum_top_k_ratio': 1.0,
   'minimum_macro_f1_ratio': 1.0,
   'maximum_false_alarm_increase_per_1000': 0.0,
   'required_guarded_high_critical_recall': 1.0}})

In [3]:
rows = []
for period_name in ("development", "temporal_holdout"):
    period = summary[period_name]
    approaches = {
        "deterministic_baseline": period["baseline"],
        **period["models"],
        "selected_statistical_workflow": period["selected_workflow"],
        "guarded_review_queue": period["guarded_queue"],
    }
    for approach, metrics in approaches.items():
        rows.append(
            {
                "period": period_name,
                "approach": approach,
                "precision": metrics["precision"],
                "recall": metrics["recall"],
                "macro_f1": metrics["macro_f1"],
                "top_50_precision": metrics["top_50_precision"],
                "false_alarms_per_1000": metrics["false_alarms_per_1000_records"],
                "high_critical_recall": metrics["high_critical_recall"],
            }
        )

comparison = pd.DataFrame(rows)
comparison.style.format(precision=3)

,period,approach,precision,recall,macro_f1,top_50_precision,false_alarms_per_1000,high_critical_recall
0,development,deterministic_baseline,0.818,1.000,0.899,0.960,3.343,1.000
1,development,isolation_forest,0.038,0.017,0.029,0.000,6.394,0.019
2,development,local_outlier_factor,0.077,0.036,0.049,0.000,6.477,0.028
3,development,selected_statistical_workflow,0.085,0.039,0.054,0.040,6.310,0.031
4,development,guarded_review_queue,0.612,1.000,0.765,0.960,9.528,1.000
5,temporal_holdout,deterministic_baseline,0.816,1.000,0.898,0.920,3.785,1.000
6,temporal_holdout,isolation_forest,0.066,0.022,0.040,0.000,5.280,0.025
7,temporal_holdout,local_outlier_factor,0.049,0.033,0.050,0.060,10.840,0.031
8,temporal_holdout,selected_statistical_workflow,0.045,0.028,0.045,0.000,9.905,0.025
9,temporal_holdout,guarded_review_queue,0.554,1.000,0.733,0.920,13.550,1.000


## Takeaways

The selected statistical workflow is retained as supplemental anomaly evidence because it does not pass the promotion gates.
The deterministic baseline remains the primary issue detector and ranking source.
The guarded review queue prevents the statistical layer from suppressing deterministic critical findings.

In [4]:
evidence_path = ARTIFACTS_DIR / "modeling" / "fy2025-bounded-anomaly-evidence.json"
evidence_preview = pd.read_json(evidence_path).head(5)
evidence_preview

,target_table,record_id,model_name,anomaly_score,rank_percentile,evidence
0,transaction,transaction:00000294,local_outlier_factor,100.0,1.0,"{""anomaly_percentile"":1.0,""anomaly_score"":100...."
1,transaction,transaction:00003973,local_outlier_factor,100.0,1.0,"{""anomaly_percentile"":1.0,""anomaly_score"":100...."
2,transaction,transaction:00003974,local_outlier_factor,100.0,1.0,"{""anomaly_percentile"":1.0,""anomaly_score"":100...."
3,transaction,transaction:00003975,local_outlier_factor,100.0,1.0,"{""anomaly_percentile"":1.0,""anomaly_score"":100...."
4,transaction,transaction:00004006,local_outlier_factor,100.0,1.0,"{""anomaly_percentile"":1.0,""anomaly_score"":100...."
